# 10: Cross-Signal Correlation & Causation Dynamics
**Project:** F1 Telemetry Intelligence  
**Grand Prix:** 2024 Abu Dhabi Grand Prix (Yas Marina Circuit)  
**Phase:** Phase 5 – Exploratory Data Analysis (EDA)  

---

## 1. Objective
Investigate linear and monotonic dependencies among kinematic sensors, evaluate multicollinearity, and distinguish correlation from causation.

## 2. Key Analytical Questions
- **Q1:** What are the linear and monotonic correlations among kinematic channels?
- **Q2:** Why does correlation fail to capture non-linear aerodynamic and braking physics?
- **Q3:** What multicollinearities exist between speed, gear, RPM, and acceleration?

## 3. Environment & Architectural Rule Compliance
- Execution Environment: Local `.venv` Python kernel in VS Code.
- Data Protection: Processed datasets are loaded strictly in read-only mode; raw data in `data/raw/` is untouched.
- Reusable Engine: Imports analytical logic directly from `src/analysis/`.


## 4. Setup & Imports

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

# Set headless matplotlib backend and temporary config
os.environ.setdefault("MPLCONFIGDIR", tempfile.gettempdir())
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Add project root to sys.path
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == "eda" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root configured: {PROJECT_ROOT}")

from src.analysis.data_loader import load_processed_telemetry
from src.analysis.correlation_analysis import (
    compute_telemetry_correlation_matrix,
    plot_correlation_heatmap,
    plot_key_signal_pairplot
)
from IPython.display import Image, display


## 5. Pearson & Spearman Correlation Matrices

In [ ]:
tel_df = load_processed_telemetry()
corr_pearson = compute_telemetry_correlation_matrix(tel_df, method="pearson", only_flying=True)
corr_spearman = compute_telemetry_correlation_matrix(tel_df, method="spearman", only_flying=True)

print("Pearson Linear Correlation Matrix:")
display(corr_pearson)


## 6. Correlation Heatmaps

In [ ]:
fig_corr_p = plot_correlation_heatmap(corr_pearson, "Pearson")
display(Image(filename=str(fig_corr_p)))

fig_corr_s = plot_correlation_heatmap(corr_spearman, "Spearman")
display(Image(filename=str(fig_corr_s)))


## 7. Pairwise Scatter Matrix of Kinematic Signals

In [ ]:
fig_pair = plot_key_signal_pairplot(tel_df, sample_size=3000)
display(Image(filename=str(fig_pair)))


## 8. Findings, Limitations & Conclusion

### Key Findings
1. **Speed & Throttle:** Strongly positively correlated ($r pprox +0.81$), reflecting engine acceleration across straights.
2. **Speed & Gear:** High positive correlation ($r pprox +0.94$), reflecting continuous upshifting with speed.
3. **Correlation vs Causation:** At maximum speed, acceleration drops to zero due to aerodynamic drag; naive linear models would misinterpret this deceleration as throttle reduction.

### Conclusion
Correlation matrix confirms expected physical couplings and highlights where non-linear models (e.g. tree ensembles or neural nets) will be required.
